# Evaluation of 3-tensor plot with real measurements

The goal is to run an experimental sampling for creating specific data points of a the probabilities 3-tensor plot. 

In [1]:
import numpy as np
import math as mt
from itertools import product
import traceback

import os
import json
import sys

from openqaoa.algorithms import QAOAResult
from openqaoa.backends import create_device
from openqaoa import QAOA

sys.path.append(os.path.abspath(os.path.join('../', 'functions')))
from utils import *
from QAOASolver import QAOASolver

In [2]:
RUN_EXPERIMENT = True

### Upload the QAOA results files

In [3]:
files = {}
directory = '../experiment_testQAOASolver/real/results'

for file in os.listdir(directory):
    if file.endswith('.json'):
        full_path = os.path.join(directory, file)
        with open(full_path, 'r', encoding='utf-8') as f:
            filename = os.path.splitext(file)[0]
            files[filename] = json.load(f)

files_keys = list(files.keys())

In [34]:
#files_keys

### Upload the 3-tensor data structures

In [4]:
axises = {}
gral_directory = '../experiment_testQAOASolver/real/results/contourf_plots'
axises_keys = [ name for name in os.listdir(gral_directory) if os.path.isdir(os.path.join(gral_directory,name)) ]

for key in axises_keys:
    directory = gral_directory + '/' + str(key) 
    axises[str(key)] = []
    for file in os.listdir(directory):
        if file.endswith('.json'):
            full_path = os.path.join(directory, file)
            with open(full_path, 'r', encoding='utf-8') as f:
                axises[str(key)].append(json.load(f))

In [36]:
#axises_keys

### Prepare data

In [37]:
# create and configure the QAOA solver for beeing used
device = create_device(location='local', name='qiskit.shot_simulator')

In order to reduce the execution time of the tests, we will define a set of constraints:

* For each pair of JRP configuration and circuit configuration, we will just work with the first sample (index 0).
* For defining the set of data points to experimentally recreate, we will take just $4$ approximation ratios and $4$ $Y = $number of shots to extrapolate, that is, a total of $4 \cdot 4 = 16$ data points. Please consider that this "number of shots to extrapolate" corresponds to the Y axis in the 3-tensor plot.
* The $4$ number of shot to extrapolate will keep the same for all the samples, BUT, the approximation ratios will varie. The idea is that approximation ratios should between the first subarray of possible ratios. That will be, the bigger ratios. This is where most of important information to validate will rely. For lower approximation ratios like $0.3$, most of the times a relatively small number of shot to extrapolate will result in a probability of measurement of $1$, which is a not so interesting estimation to validate.

In [38]:
q_solver = QAOASolver()
results = {} # data structure for storing the results of the experiment
sample_index = 0 # the sample used from each JRP and circuit configurations will be #1
n_xvalues = 4 # number of approximation ratios to be tested.
n_zvalues = 4 # number of different number of shots to be tested. For each JRP-circuit conf, there will be n_xvalues * n_zvalues results.
DEBUG =False

# for each file, that is, each JRP-circuit conf. we will initialize the result structure for sample 1
for current_filename,current_file in files.items():
    # get the circuit configuration and corresponding sample 1
    current_circuit_configuration = current_file['circuit_configuration']
    current_sample = current_file[str(sample_index)]

    # create the JRP ising
    current_instance_dict = current_sample["instance"]
    current_jrp = JRPClassic(current_instance_dict)
    current_jrp_ising = q_solver.get_jrp_ising(current_jrp)
    
    # calculate standard gain
    current_optimal_standard_solution_gain = current_jrp.calculate_standard_gain(current_sample['opt_standard_solution'])

    # get 3-tensor data structure of theoretical probs. to compare.
    current_axises = axises[current_filename][sample_index]

    # make lists into np array
    #current_axises['X'] = np.array(current_axises['X'])
    #current_axises['Z'] = np.array(current_axises['Z'])
    current_axises['Y'] = np.array(current_axises['Y'])

    if DEBUG:
        print('Axises for',current_filename)
        print('Shape of approximation ratios:',current_axises['X'].shape)
        print('Shape of n measurements:',current_axises['Z'].shape)
        print('Shape of probabilities:',current_axises['Y'].shape,'\n')

    # define a set of approximation ratios indices to try.
    # they should be in the 15% low half of positions, since it is where more interesting probabilities will be seen (that is, probs in [0.3,0.9])
    upper_xindex_value = round(current_axises['Y'].shape[1] * 0.15)
    shift = mt.ceil(upper_xindex_value / n_xvalues)
    xindexes_to_test = [i for i in range(0,upper_xindex_value,shift)]

    current_axises['Y'] = current_axises['Y'].tolist()

    
    # initialize the result info. It still needs to be updated in further cells.
    results[current_filename] = {
        'circuit_configuration':current_circuit_configuration,
        'sample_used':sample_index,
        
        'jrp':current_jrp,
        'jrp_ising':current_jrp_ising,
        'optimal_standard_solution_gain':current_optimal_standard_solution_gain,

        'axises':current_axises,
        'xindexes_to_test':xindexes_to_test,
        
        'approximation_ratios':[],
        'n_measurements':[],
        
        'theoretical_probabilities_for_xMinAproxRatio_zNMeasurements':{},
        'experimental_probabilities_for_xMinAproxRatio_zNMeasurements':{}
    }

### Run experiment

Before running the experiment, we define the following function. This makes the experimental sampling for a particular JRP instance.

In [40]:
def experimental_tensor3_for_approximation_ratio(results_structure,resultname,n_samples,
                                                 current_axises,xindexes_to_test,
                                                 zindexes_to_test,jrp,jrp_ising,optimal_standard_solution_gain):
    '''
    '''

    # data structures with the theoretical probabilities and the experimental probabilities.
    # they will be used to be compared and see how accurate is the theoretical estimation.
    theoretical_probabilities_for_xMinAproxRatio_zNMeasurements = {}
    experimental_probabilities_for_xMinAproxRatio_zNMeasurements = {}

    for ix,iz in product(xindexes_to_test,zindexes_to_test):
        # current aprox. ratio and n. of measurements.
        x = current_axises['X'][ix]
        z = current_axises['Z'][iz]

        print(f'({z},{x})')

        # total shots to run. That is, the product of number of measurements and the number of samples of shotings to run.
        total_shots = z * n_samples
        #print(f'{resultname}:  {str(ix*iz+iz)}/{total_shots}  -   x{ix},z{iz}: ({x}, {z})',end='\r')
        
        # configure QAOA
        qaoa = QAOA()
        qaoa.set_device(device)
        qaoa.set_circuit_properties(**current_circuit_configuration)
        qaoa.set_backend_properties(n_shots= total_shots)
        qaoa.compile(jrp_ising)

        
        # get the solutions, shuffle them and split them in different samples
        solutions_bitstring = qaoa.evaluate_circuit(current_sample['result']['optimized']['angles'])['measurement_results']  
        solutions_bitstring = np.array([label for label, count in solutions_bitstring.items() for _ in range(count)]) 
        np.random.shuffle(solutions_bitstring) 
        solutions_bitstring = np.array_split(solutions_bitstring,n_samples)

        probability_measuring = 0
        for sample,sub_solutions in enumerate(solutions_bitstring):             
            # convert solutions to standard formulation
            #print(sub_solutions.shape)
            sub_solutions = sub_solutions.tolist()
            #print(sub_solutions)
            standard_solutions = []
            approximation_ratios = []
            for s in sub_solutions:
                try:
                    # if the method raise an error, means that the solution is unfeasible, so it won't be appended to final list
                    ss=jrp.quboSolution_to_standardSolution(s,check_feasibility=True)
                    standard_solutions.append(ss)
                    approx_ratio = jrp.calculate_standard_gain(ss)/optimal_standard_solution_gain
                    approximation_ratios.append(round(approx_ratio,4))
                except Exception as e:
                    pass

            # See if at least the largest approx ratio achieves 'x' as min. value
            approximation_ratios.sort() 
            #print(approximation_ratios[-1])
            if len(approximation_ratios) > 0 and approximation_ratios[-1] >= x:
                probability_measuring += 1
        # FINAL PROBABILITY TO STORE
        probability_measuring = probability_measuring / n_samples
        

        # update the theoretical and experimental probs. structures.
        theoretical_probabilities_for_xMinAproxRatio_zNMeasurements[str((z,x100list[itrx100]))] = current_axises['Y'][iz][current_axises['X'].index(x100list[itrx100])]
        experimental_probabilities_for_xMinAproxRatio_zNMeasurements[str((z,x))] = probability_measuring
    
        # add results dats to the the result structre
        results_structure[resultname].update({
        'zindexes_to_test':zindexes_to_test,

        'n_measurements':zindexes_to_test,
        
        'theoretical_probabilities_for_xMinAproxRatio_zNMeasurements':theoretical_probabilities_for_xMinAproxRatio_zNMeasurements,
        'experimental_probabilities_for_xMinAproxRatio_zNMeasurements':experimental_probabilities_for_xMinAproxRatio_zNMeasurements
        })   
        results_structure[resultname]['approximation_ratios'].append(x100list[itrx100])#results_structure[resultname]['axises']['X'][ix])

        with open('results/100_sampling/'+resultname+'.json', "w", encoding="utf-8") as file:
            json.dump( results_structure[resultname], file, indent=4, ensure_ascii=False)
        itrx100 +=1
    print(f'{resultname}:{theoretical_probabilities_for_xMinAproxRatio_zNMeasurements}')

Finally, we run the experiment. Lets describe it:

The theoretical 3-tensor data structure is formed by data points $(X,Y,Z)$ which represent that a min. aprox. ratio of $X$ is achieved with $Y$ measurement shots with a probability of $Z$. For experimentally calculating a data point $(X,Y,Z)$ we need to run $Y \cdot N$ measurement shots. The variable $N$ represents the number of samples for estimating $Z$, so that each sample $m \in M$ has $Y$ measurement shots. Doing this, $Z$ is the reason between the number of samples in $M$ where a solution with a min. approx ratio $X$ is found and $\#M$.


In [45]:
zindexes_to_test = [10,100,500,750]
n_samples=100 # The M variable mentiones

for resultname,result in results.items():
    axises=result['axises']
    xindexes_to_test = result['xindexes_to_test']
    jrp =  result['jrp']
    jrp_ising = result['jrp_ising']
    optimal_standard_solution_gain = result['optimal_standard_solution_gain']

    del results[resultname]['jrp']
    del results[resultname]['jrp_ising']
    
    try:
        experimental_tensor3_for_approximation_ratio(results,resultname,n_samples,axises,xindexes_to_test,zindexes_to_test,jrp,jrp_ising,optimal_standard_solution_gain)
    except FileNotFoundError:
        print(f"El archivo {resultname}.json no existe. Se omite.")

x100list[1.0, 1.0, 1.0, 1.0, 0.9772, 0.9772, 0.9772, 0.9772, 0.9566, 0.9566, 0.9566, 0.9566, 0.9224, 0.9224, 0.9224, 0.9224]
(10,1.0)
(100,1.0)
(500,1.0)
(750,1.0)
(10,0.9772)
(100,0.9772)
(500,0.9772)
(750,0.9772)
(10,0.9566)
(100,0.9566)
(500,0.9566)
(750,0.9566)
(10,0.9224)
(100,0.9224)
(500,0.9224)
(750,0.9224)
conf_jrpInitConf0_circuitConf0:{'(10, 1.0)': 0.024720616821432673, '(100, 1.0)': 0.22144296041027722, '(500, 1.0)': 0.713943263669631, '(750, 1.0)': 0.8470046367476098, '(10, 0.9772)': 0.14809274249551518, '(100, 0.9772)': 0.7986631912215345, '(500, 0.9772)': 0.9996691616063759, '(750, 0.9772)': 0.9999939823925301, '(10, 0.9566)': 0.25417052732647516, '(100, 0.9566)': 0.9467407051357103, '(500, 0.9566)': 0.9999995714741144, '(750, 0.9566)': 0.9999999997194788, '(10, 0.9224)': 0.3596820531142134, '(100, 0.9224)': 0.9884133805204434, '(500, 0.9224)': 0.9999999997911744, '(750, 0.9224)': 0.999999999999997}
x100list[1.0, 1.0, 1.0, 1.0, 0.9772, 0.9772, 0.9772, 0.9772, 0.9566, 0.9